In [2]:
from pyspark.sql import SparkSession 

spark = (
    SparkSession.builder
    .appName("Weather_Analytics_Pipeline")
    .getOrCreate()
)
spark 

### Reading the Raw JSON File ### 

In [3]:
df = spark.read.option("multiline","true").json("../Data/01_Bronze/weather_hourly_raw.json") 

In [4]:
df.printSchema() 

root
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- hourly: struct (nullable = true)
 |    |-- relative_humidity_2m: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- temperature_2m: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- hourly_units: struct (nullable = true)
 |    |-- relative_humidity_2m: string (nullable = true)
 |    |-- temperature_2m: string (nullable = true)
 |    |-- time: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = true)
 |-- utc_offset_seconds: long (nullable = true)



In [5]:
df.select("hourly.*").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
from pyspark.sql.functions import * 

zipped_df = df.select(
    arrays_zip(
        "hourly.time",
        "hourly.temperature_2m",
        "hourly.relative_humidity_2m"
    ).alias("weather_data")
) 

In [9]:
exploded_df = zipped_df.select(explode("weather_data").alias("weather")) 

In [10]:
exploded_df.show(5,truncate=False) 

+----------------------------+
|weather                     |
+----------------------------+
|{2026-06-09T00:00, 30.7, 66}|
|{2026-06-09T01:00, 30.6, 65}|
|{2026-06-09T02:00, 31.3, 63}|
|{2026-06-09T03:00, 32.4, 58}|
|{2026-06-09T04:00, 33.9, 52}|
+----------------------------+
only showing top 5 rows


In [11]:
weather_df = exploded_df.select(
    "weather.time",
    "weather.temperature_2m",
    "weather.relative_humidity_2m"
) 

In [12]:
weather_df.show(10, truncate=False) 

+----------------+--------------+--------------------+
|time            |temperature_2m|relative_humidity_2m|
+----------------+--------------+--------------------+
|2026-06-09T00:00|30.7          |66                  |
|2026-06-09T01:00|30.6          |65                  |
|2026-06-09T02:00|31.3          |63                  |
|2026-06-09T03:00|32.4          |58                  |
|2026-06-09T04:00|33.9          |52                  |
|2026-06-09T05:00|35.3          |47                  |
|2026-06-09T06:00|36.9          |42                  |
|2026-06-09T07:00|37.7          |41                  |
|2026-06-09T08:00|39.1          |37                  |
|2026-06-09T09:00|39.9          |34                  |
+----------------+--------------+--------------------+
only showing top 10 rows


In [18]:
weather_df = weather_df.select(
    col("time").alias("weather_time"),
    col("temperature_2m").alias("temperature"),
    col("relative_humidity_2m").alias("humidity")
)

In [19]:
weather_df = weather_df.withColumn(
    "weather_time",to_timestamp("weather_time") 
)

In [23]:
weather_df = (
    weather_df
    .withColumn("weather_date",
                to_date("weather_time"))
    .withColumn("weather_hour",
                hour("weather_time"))
)

In [25]:
weather_df = weather_df.withColumn(
    "ingestion_date",current_timestamp()
) 

### Data Quality Check ### 

In [26]:
# CHECKING THE TOTAL RECORDS 
weather_df.count() 

336

In [27]:
# CHECKING FOR THE NULL VALUES  
weather_df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in weather_df.columns
]).show() 

+------------+-----------+--------+------------+------------+--------------+
|weather_time|temperature|humidity|weather_date|weather_hour|ingestion_date|
+------------+-----------+--------+------------+------------+--------------+
|           0|          0|       0|           0|           0|             0|
+------------+-----------+--------+------------+------------+--------------+



In [28]:
#BASIC STATISTICS 
weather_df.describe().show() 

+-------+------------------+-----------------+-----------------+
|summary|       temperature|         humidity|     weather_hour|
+-------+------------------+-----------------+-----------------+
|  count|               336|              336|              336|
|   mean| 34.13095238095236|52.24702380952381|             11.5|
| stddev|3.5271053182579983|14.25927265983405|6.932510475472596|
|    min|              29.1|               25|                0|
|    max|              42.1|               78|               23|
+-------+------------------+-----------------+-----------------+



In [29]:
weather_df.write.mode("overwrite").parquet(
    "../Data/02_Silver/weather_cleaned"
)